In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
df = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper to compute aggregations for metric-like files
def summarize_timeseries(df_in, id_col, name_col, value_col='value', ts_col='timestamp', top_n=50):
    g = df_in.groupby([id_col, name_col])
    agg_vals = g[value_col].agg(
        count='count',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p75=lambda x: x.quantile(0.75),
        p95=lambda x: x.quantile(0.95)
    )
    agg_ts = g[ts_col].agg(min_ts='min', max_ts='max')
    summary = pd.concat([agg_vals, agg_ts], axis=1).reset_index()
    # Round numeric summary columns for compactness
    for c in ['mean', 'p50', 'p75', 'p95']:
        if c in summary.columns:
            summary[c] = summary[c].round(6)
    summary = summary.sort_values('count', ascending=False).head(top_n)
    return summary

# 1) metric.csv summary (up to 50 rows)
summary_metric = summarize_timeseries(df, 'cmdb_id', 'kpi_name')

# 2) trace.csv summary (up to 50 rows)
summary_trace = summarize_timeseries(df_trace, 'cmdb_id', 'trace_name')

# 3) log.csv summary (up to 50 rows)
summary_log = summarize_timeseries(df_log, 'cmdb_id', 'log_name')

# 4) error_logs.csv compact summary
err_total = df_err.shape[0]
err_cmdb_counts = df_err['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
err_ts_min = df_err['timestamp'].min() if err_total>0 else np.nan
err_ts_max = df_err['timestamp'].max() if err_total>0 else np.nan

# Display compact summaries (variables shown will be the output)
summary_metric, summary_trace, summary_log, err_total, err_cmdb_counts, err_ts_min, err_ts_max

```
Out[1]:
```
summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    "- Returned up to 50 unique (cmdb_id, kpi_name) pairs with aggregates (count, mean, p50, p75, p95, min_ts, max_ts).\n"
    "- Top pairs have count = 31 (many Tomcat01 and Mysql01/Mysql02 KPIs). Many entries show min_ts=1615251600 and max_ts=1615253400.\n\n"
    "trace.csv:\n"
    "- Returned up to 50 unique (cmdb_id, trace_name) pairs with aggregates.\n"
    "- Top pairs have count = 30 (examples: IG01.trace.self.duration_mean). Reported min_ts=1615251600 and max_ts=1615253340 for the top results.\n\n"
    "log.csv:\n"
    "- Returned up to 50 unique (cmdb_id, log_name) pairs with aggregates.\n"
    "- Top counts are 31 for several entries (e.g., Tomcat01 log.row_count and log.error_count, apache01/apache02 log.row_count). Some components have counts = 30 or 18.\n"
    "- min_ts values are mostly 1615251600; max_ts are 1615253400 or 1615253340 depending on the series.\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {err_total}\n"
    f"- Top cmdb_id counts (up to 20 shown):\n{err_cmdb_counts.to_string(index=False)}\n"
    f"- Earliest timestamp: {int(err_ts_min) if not pd.isna(err_ts_min) else err_ts_min}\n"
    f"- Latest timestamp: {int(err_ts_max) if not pd.isna(err_ts_max) else err_ts_max}\n\n"
    "If you want, I can display the top N rows of any of the summary tables (metric/trace/log) for more detail."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           mean            p50            p75           p95      min_ts      max_ts
1183  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31       1.677419       1.000000       3.000000       5.00000  1615251600  1615253400
1169  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31     642.000000     642.000000     642.000000     642.00000  1615251600  1615253400
1170  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  405396.000000  405396.000000  405396.000000  405396.00000  1615251600  1615253400
499    Mysql01                        OSLinux-CPU_CPU_CPUidleutil     31      98.213949      98.570122      98.684200      98.86938  1615251600  1615253400
498    Mysql01                             OSLinux-CPU_CPU_CPUWio     31       1.110612       0.747912       1.282282       2.48285  1615251600  1615253400
...        ...                                                ...    ...            ...            ...            ...           ...         ...         ...
720    Mysql02                         OSLinux-CPU_CPU_CPUSysTime     31       1.239337       1.241600       1.344966       1.47565  1615251600  1615253400
495    Mysql01                            OSLinux-CPU_CPU_CPULoad     31       0.112581       0.070000       0.165000       0.35000  1615251600  1615253400
1178  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31       0.000000       0.000000       0.000000       0.00000  1615251600  1615253400
1179  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31       0.000000       0.000000       0.000000       0.00000  1615251600  1615253400
1180  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31       0.000000       0.000000       0.000000       0.00000  1615251600  1615253400

[50 rows x 9 columns],    cmdb_id                       trace_name  count         mean          p50          p75          p95      min_ts      max_ts
0     IG01         trace.self.duration_mean     30     4.132895     4.097669     4.327477     4.505261  1615251600  1615253340
1     IG01          trace.self.duration_p95     30    15.559745    15.558200    15.569412    15.579885  1615251600  1615253340
2     IG01             trace.self.row_count     30  1390.866667  1347.000000  1502.000000  1669.300000  1615251600  1615253340
3     IG01  trace.to_Tomcat01.duration_mean     30     0.299835     0.287316     0.344395     0.502944  1615251600  1615253340
4     IG01   trace.to_Tomcat01.duration_p95     30     1.523337     1.292875     2.055550     2.136272  1615251600  1615253340
..     ...                              ...    ...          ...          ...          ...          ...         ...         ...
45    MG01         trace.self.duration_mean     30     0.170561     0.166921     0.190612     0.255097  1615251600  1615253340
46    MG01          trace.self.duration_p95     30     1.123458     1.067550     1.088462     1.399080  1615251600  1615253340
47    MG01             trace.self.row_count     30   778.733333   766.500000   840.750000   966.550000  1615251600  1615253340
48    MG01  trace.to_dockerA1.duration_mean     30     0.176063     0.181200     0.223952     0.270401  1615251600  1615253340
49    MG01   trace.to_dockerA1.duration_p95     30     1.005592     1.031300     1.037875     1.097287  1615251600  1615253340

[50 rows x 9 columns],      cmdb_id          log_name  count        mean     p50      p75       p95      min_ts      max_ts
4   Tomcat01     log.row_count     31  337.419355  342.00  370.000  410.0000  1615251600  1615253400
2   Tomcat01   log.error_count     31    0.000000    0.00    0.000    0.0000  1615251600  1615253400
17  apache02     log.row_count     31  676.161290  673.00  736.500  842.5000  1615251600  1615253400
15  apache01     log.row_count     31  682.451613  698.00  748.500  850.5000  1615251600  1615253400
8   Tomcat03   log.error_count     31    0.000000    0.00    0.000    0.0000  1615251600  1615253400
16  apache02   log.error_count     31    0.032258    0.00    0.000    0.0000  1615251600  1615253400
14  apache01   log.error_count     31    0.032258    0.00    0.000    0.0000  1615251600  1615253400
13  Tomcat04     log.row_count     31  339.096774  348.00  367.500  411.5000  1615251600  1615253400
11  Tomcat04   log.error_count     31    0.000000    0.00    0.000    0.0000  1615251600  1615253400
10  Tomcat03     log.row_count     31  343.161290  350.00  377.500  441.5000  1615251600  1615253400
6   Tomcat02  log.gc_pause_max     30    2.010000    1.94    1.995    2.3575  1615251600  1615253340
7   Tomcat02     log.row_count     30   35.700000   36.00   38.000   39.0000  1615251600  1615253340
5   Tomcat02   log.error_count     30    0.033333    0.00    0.000    0.0000  1615251600  1615253340
0    Mysql01   log.error_count     18    0.000000    0.00    0.000    0.0000  1615251600  1615253400
1    Mysql01     log.row_count     18    4.333333    3.50    4.750   10.2500  1615251600  1615253400
3   Tomcat01  log.gc_pause_max     18    0.060556    0.06    0.080    0.0900  1615251600  1615253280
9   Tomcat03  log.gc_pause_max     18    0.051667    0.06    0.070    0.0800  1615251660  1615253340
12  Tomcat04  log.gc_pause_max     18    0.060000    0.07    0.070    0.0915  1615251660  1615253340, 3,     cmdb_id  count
0  Tomcat02      1
1  apache01      1
2  apache02      1, np.int64(1615252644), np.int64(1615252670))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse previously loaded DataFrames if available; otherwise load from CSV
try:
    df, df_trace, df_log, df_err
except NameError:
    df = pd.read_csv('metric.csv')
    df_trace = pd.read_csv('trace.csv')
    df_log = pd.read_csv('log.csv')
    df_err = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')

# Allowed cmdb list
allowed_cmdbs = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                 "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Helper aggregator function
def compute_aggs(df_in, group_cols, value_col='value', ts_col='timestamp'):
    g = df_in.groupby(group_cols)
    agg_vals = g[value_col].agg(
        count='count',
        p5=lambda x: x.quantile(0.05),
        p50='median',
        p95=lambda x: x.quantile(0.95)
    )
    agg_ts = g[ts_col].agg(min_ts='min', max_ts='max')
    out = pd.concat([agg_vals, agg_ts], axis=1).reset_index()
    # Round numeric columns for compactness
    for c in ['p5','p50','p95']:
        if c in out.columns:
            out[c] = out[c].round(6)
    return out

# 1) metric.csv aggregations (global thresholds)
metric_filtered = df[df['cmdb_id'].isin(allowed_cmdbs)]
metric_aggs = compute_aggs(metric_filtered, ['cmdb_id','kpi_name'], 'value', 'timestamp')
metric_aggs_top20 = metric_aggs.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','kpi_name','count','p5','p50','p95','min_ts','max_ts']
]

# 2) trace.csv aggregations
trace_filtered = df_trace[df_trace['cmdb_id'].isin(allowed_cmdbs)]
trace_aggs = compute_aggs(trace_filtered, ['cmdb_id','trace_name'], 'value', 'timestamp')
trace_aggs_top20 = trace_aggs.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','trace_name','count','p5','p50','p95','min_ts','max_ts']
]

# 3) log.csv aggregations
log_filtered = df_log[df_log['cmdb_id'].isin(allowed_cmdbs)]
log_aggs = compute_aggs(log_filtered, ['cmdb_id','log_name'], 'value', 'timestamp')
log_aggs_top20 = log_aggs.sort_values('count', ascending=False).head(20)[
    ['cmdb_id','log_name','count','p5','p50','p95','min_ts','max_ts']
]

# 4) error_logs.csv: distinct cmdb_id counts (full) and up to 50 rows
err_cmdb_counts_full = df_err['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
err_rows_limited = df_err[['timestamp','cmdb_id','message']].head(50)
err_ts_min = df_err['timestamp'].min() if not df_err.empty else np.nan
err_ts_max = df_err['timestamp'].max() if not df_err.empty else np.nan

# Display compact outputs (top summaries)
metric_aggs_top20, trace_aggs_top20, log_aggs_top20, err_cmdb_counts_full, err_rows_limited, err_ts_min, err_ts_max

```
Out[2]:
```
Summary (plain English):

- metric.csv (global aggregates, top 20 groups):
  - Top groups mostly for Tomcat and apache components with count = 31 per series.
  - Many Tomcat Request and Processing KPIs show very large numeric values (examples: RequestCount p50 ~1.17e6, Processing p50 ~7.25e8), while many Tomcat Sessions KPIs are zero (p5/p50/p95 = 0).
  - Some OS metrics for apache (e.g., OSLinux CPU load) show small values (example apache02 OSLinux-CPU_CPU_CPULoad p50 = 0.01, p95 = 0.1).
  - All top metric series span min_ts = 1615251600 and max_ts = 1615253400.

- trace.csv (global aggregates, top 20 groups):
  - Top trace groups are from IG01 and IG02 with count = 30.
  - Latency/trace summaries: trace.self.duration_mean p50 ≈ 4.0977, trace.self.duration_p95 p50 ≈ 15.55.
  - Row counts for IG services: p50 ≈ 1347 (IG01) and similar for IG02.
  - Inter-service call durations to Tomcat nodes have p50 in the ~0.13–0.33 range and p95 up to ~2.28 for some calls.
  - Trace series span min_ts = 1615251600 and max_ts = 1615253340.

- log.csv (global aggregates, top 20 groups):
  - Many log.row_count series for Tomcat and apache have count = 31; typical p50 values: Tomcat log.row_count ~342, apache log.row_count ~673–698.
  - log.error_count series mostly have p50 = 0 (very few/no errors per minute in these aggregates).
  - Some gc_pause_max series present (e.g., Tomcat02 p50 ≈ 1.94, Tomcat01 p50 ≈ 0.06).
  - Log series min_ts mostly 1615251600 and max_ts 1615253400 (some end at 1615253340 or 1615253280 for shorter series).

- error_logs.csv:
  - Total distinct error rows shown: 3 rows (one per listed sample).
  - Distinct cmdb_id counts (full): Tomcat02: 1, apache01: 1, apache02: 1.
  - Earliest timestamp in the error sample: 1615252644. Latest timestamp: 1615252670.
  - The displayed error rows (up to 50) include one error each for Tomcat02, apache01, apache02.

Notes:
- The full aggregated tables are stored in kernel variables metric_aggs, trace_aggs, and log_aggs for further analysis if you want to query or filter them.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           p5           p50          p95      min_ts      max_ts
1320  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31        558.0  5.580000e+02        558.0  1615251600  1615253400
1335  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1334  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1333  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1332  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1331  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1330  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1327  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1326  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1325  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1324  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31          0.0  0.000000e+00          0.0  1615251600  1615253400
1537  apache02                            OSLinux-CPU_CPU_CPULoad     31          0.0  1.000000e-02          0.1  1615251600  1615253400
1323  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31    1166226.0  1.170633e+06    1175734.0  1615251600  1615253400
1520  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31          0.4  5.000000e-01          0.5  1615251600  1615253400
1521  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31          4.0  4.000000e+00          4.0  1615251600  1615253400
1527  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31    6450393.0  6.451203e+06    6452013.0  1615251600  1615253400
1322  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  724085742.5  7.254476e+08  727220750.0  1615251600  1615253400
1125  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31        642.0  6.420000e+02        642.0  1615251600  1615253400
1126  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31     405396.0  4.053960e+05     405396.0  1615251600  1615253400
1127  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  757377196.0  7.587225e+08  760309284.5  1615251600  1615253400,    cmdb_id                       trace_name  count           p5          p50          p95      min_ts      max_ts
0     IG01         trace.self.duration_mean     30     3.740345     4.097669     4.505261  1615251600  1615253340
1     IG01          trace.self.duration_p95     30    15.531100    15.558200    15.579885  1615251600  1615253340
2     IG01             trace.self.row_count     30  1231.000000  1347.000000  1669.300000  1615251600  1615253340
3     IG01  trace.to_Tomcat01.duration_mean     30     0.163386     0.287316     0.502944  1615251600  1615253340
4     IG01   trace.to_Tomcat01.duration_p95     30     1.044070     1.292875     2.136272  1615251600  1615253340
5     IG01      trace.to_Tomcat01.row_count     30   142.800000   168.500000   211.500000  1615251600  1615253340
6     IG01  trace.to_Tomcat03.duration_mean     30     0.171970     0.313855     0.483363  1615251600  1615253340
7     IG01   trace.to_Tomcat03.duration_p95     30     1.057140     1.434875     2.284510  1615251600  1615253340
8     IG01      trace.to_Tomcat03.row_count     30   141.450000   171.000000   208.400000  1615251600  1615253340
9     IG01  trace.to_Tomcat04.duration_mean     30     0.167967     0.296322     0.508816  1615251600  1615253340
10    IG01   trace.to_Tomcat04.duration_p95     30     1.058608     1.190950     2.131032  1615251600  1615253340
11    IG01      trace.to_Tomcat04.row_count     30   142.800000   174.000000   206.400000  1615251600  1615253340
12    IG02         trace.self.duration_mean     30     3.699271     4.058340     4.369410  1615251600  1615253340
13    IG02          trace.self.duration_p95     30    15.512092    15.544750    15.568000  1615251600  1615253340
14    IG02             trace.self.row_count     30  1207.000000  1413.000000  1703.700000  1615251600  1615253340
15    IG02  trace.to_Tomcat01.duration_mean     30     0.136396     0.272569     0.503088  1615251600  1615253340
16    IG02   trace.to_Tomcat01.duration_p95     30     1.037045     1.145750     2.163358  1615251600  1615253340
17    IG02      trace.to_Tomcat01.row_count     30   142.900000   175.000000   212.150000  1615251600  1615253340
18    IG02  trace.to_Tomcat03.duration_mean     30     0.177063     0.329244     0.525936  1615251600  1615253340
19    IG02   trace.to_Tomcat03.duration_p95     30     1.056815     1.241900     2.218027  1615251600  1615253340,      cmdb_id          log_name  count      p5     p50       p95      min_ts      max_ts
4   Tomcat01     log.row_count     31  292.00  342.00  410.0000  1615251600  1615253400
2   Tomcat01   log.error_count     31    0.00    0.00    0.0000  1615251600  1615253400
17  apache02     log.row_count     31  601.50  673.00  842.5000  1615251600  1615253400
15  apache01     log.row_count     31  596.50  698.00  850.5000  1615251600  1615253400
8   Tomcat03   log.error_count     31    0.00    0.00    0.0000  1615251600  1615253400
16  apache02   log.error_count     31    0.00    0.00    0.0000  1615251600  1615253400
14  apache01   log.error_count     31    0.00    0.00    0.0000  1615251600  1615253400
13  Tomcat04     log.row_count     31  296.50  348.00  411.5000  1615251600  1615253400
11  Tomcat04   log.error_count     31    0.00    0.00    0.0000  1615251600  1615253400
10  Tomcat03     log.row_count     31  285.00  350.00  441.5000  1615251600  1615253400
6   Tomcat02  log.gc_pause_max     30    1.90    1.94    2.3575  1615251600  1615253340
7   Tomcat02     log.row_count     30   32.00   36.00   39.0000  1615251600  1615253340
5   Tomcat02   log.error_count     30    0.00    0.00    0.0000  1615251600  1615253340
0    Mysql01   log.error_count     18    0.00    0.00    0.0000  1615251600  1615253400
1    Mysql01     log.row_count     18    1.00    3.50   10.2500  1615251600  1615253400
3   Tomcat01  log.gc_pause_max     18    0.02    0.06    0.0900  1615251600  1615253280
9   Tomcat03  log.gc_pause_max     18    0.02    0.06    0.0800  1615251660  1615253340
12  Tomcat04  log.gc_pause_max     18    0.02    0.07    0.0915  1615251660  1615253340,     cmdb_id  count
0  Tomcat02      1
1  apache01      1
2  apache02      1,     timestamp   cmdb_id                                            message
0  1615252644  Tomcat02  IG02 POST /UOCP/base/ServiceTest4.json HTTP/1....
1  1615252668  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso...
2  1615252670  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H..., np.int64(1615252644), np.int64(1615252670))```
```

In [ ]:
# In[3]:

import numpy as np
import pytz
from datetime import datetime

# Reuse existing DataFrames: df (metric.csv) and metric_aggs
# If they are not present, load them (safe-guard)
try:
    df
except NameError:
    import pandas as pd
    df = pd.read_csv('metric.csv')

try:
    metric_aggs
except NameError:
    # If metric_aggs not present, compute quickly for full df (but instruction expects metric_aggs exists)
    def _compute_aggs_quick(df_in):
        g = df_in.groupby(['cmdb_id','kpi_name'])
        agg_vals = g['value'].agg(count='count',
                                  p5=lambda x: x.quantile(0.05),
                                  p50='median',
                                  p95=lambda x: x.quantile(0.95))
        agg_ts = g['timestamp'].agg(min_ts='min', max_ts='max')
        out = pd.concat([agg_vals, agg_ts], axis=1).reset_index()
        for c in ['p5','p50','p95']:
            if c in out.columns:
                out[c] = out[c].round(6)
        return out
    metric_aggs = _compute_aggs_quick(df)

# 1) Incident window in UTC+8: Mar 9, 2021 09:00 to 09:30 inclusive
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021,3,9,9,0,0))
end_dt = tz.localize(datetime(2021,3,9,9,30,0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# 1) Filter metric.csv to incident window (inclusive)
metric_incident = df[(df['timestamp'] >= start_ts) & (df['timestamp'] <= end_ts)].copy()

# 2) Join metric_incident with metric_aggs on (cmdb_id, kpi_name) to bring p5 and p95
# Ensure metric_aggs contains p5 and p95 columns
merge_cols = ['cmdb_id','kpi_name']
metric_merged = metric_incident.merge(
    metric_aggs[merge_cols + ['p5','p95']],
    on=merge_cols,
    how='left'
)

# 3) Compute boolean flags (treat missing p5/p95 as non-anomalous naturally via comparisons)
metric_merged['high_anom'] = metric_merged['value'] > metric_merged['p95']
metric_merged['low_anom']  = metric_merged['value'] < metric_merged['p5']

# 4) Aggregate anomalies by (cmdb_id, kpi_name)
# Consider only rows that are anomalous (either high or low)
metric_anom_rows = metric_merged[metric_merged['high_anom'] | metric_merged['low_anom']].copy()

if not metric_anom_rows.empty:
    agg = metric_anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        count_anom = ('value','size'),
        earliest_anom_ts = ('timestamp','min'),
        max_value = ('value','max'),
        min_value = ('value','min'),
        any_high = ('high_anom','any'),
        any_low  = ('low_anom','any'),
        p95 = ('p95','first'),
        p5  = ('p5','first')
    ).reset_index()
    # Determine anomaly_type
    def _atype(row):
        if row['any_high']:
            return 'high'
        if row['any_low']:
            return 'low'
        return 'none'
    agg['anomaly_type'] = agg.apply(_atype, axis=1)
    # Compute severity with safe divide (set to NaN if divisor is zero or NaN)
    def _severity(row):
        if row['anomaly_type'] == 'high':
            denom = row['p95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['max_value'] / denom
        elif row['anomaly_type'] == 'low':
            denom = row['min_value']
            numer = row['p5']
            if pd.isna(denom) or denom == 0 or pd.isna(numer):
                return np.nan
            return numer / denom
        else:
            return np.nan
    agg['severity'] = agg.apply(_severity, axis=1)
    # Clean up and select required columns
    metric_anoms = agg[['cmdb_id','kpi_name','count_anom','earliest_anom_ts','anomaly_type','max_value','severity','p5','p95','min_value']]
else:
    # empty DataFrame with expected columns
    metric_anoms = pd.DataFrame(columns=['cmdb_id','kpi_name','count_anom','earliest_anom_ts','anomaly_type','max_value','severity','p5','p95','min_value'])

# 5) Return compact top-20 table sorted by count_anom desc then severity desc
metric_anoms_sorted = metric_anoms.sort_values(['count_anom','severity'], ascending=[False, False])
metric_anoms_top20 = metric_anoms_sorted[['cmdb_id','kpi_name','count_anom','anomaly_type','earliest_anom_ts','max_value','severity']].head(20)

# Additional counts
total_series = metric_incident[['cmdb_id','kpi_name']].drop_duplicates().shape[0]
anomalous_series_count = metric_anoms.shape[0]

# Keep metric_incident and metric_anoms in kernel for next steps
metric_incident, metric_anoms, metric_anoms_top20, total_series, anomalous_series_count

```
Out[3]:
```
Summary of metric-level anomaly detection (incident window Mar 9, 2021 09:00–09:30 UTC+8):

- Data filtered for the incident window:
  - metric_incident rows: 27,468 (all metric rows whose timestamp is between 1615251600 and 1615253400 inclusive).
  - Distinct (cmdb_id, kpi_name) series in window: 1,688.

- Anomalies aggregated (metric_anoms):
  - Series with at least one anomaly: 658 distinct (cmdb_id, kpi_name) groups.
  - metric_anoms was computed and stored in kernel (variable name: metric_anoms).
  - metric_incident is also stored in kernel (variable name: metric_incident).

- How anomalies were flagged and severity computed:
  - Each incident-row compared to global thresholds (p5 and p95 from metric_aggs).
  - high_anom if value > p95; low_anom if value < p5. Missing p5/p95 treated as non-anomalous for that row.
  - Aggregation per series produced count_anom, earliest_anom_ts, max_value (and min_value), and anomaly_type ('high' if any high_anom else 'low' if any low_anom else 'none').
  - Severity: for 'high' series severity = max_value / p95; for 'low' series severity = p5 / min_value. Division-by-zero or missing denominators result in NaN severity.

- Top anomaly observations (compact):
  - Many filesystem capacity series for multiple nodes show repeated anomalies (example top counts: IG01 and IG02 OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity each had count_anom = 9 and anomaly_type = low with severity = 1.0).
  - Several CPU-related metrics exhibited high anomalies across nodes (examples with count_anom = 4: various OSLinux CPU user/sys/wio metrics for Mysql02, Redis01/Redis02, apache02, etc.; many have severity > 1 indicating values exceeded the p95 threshold).
  - Memory/JVM metrics on IG01 show high anomalies (examples: JVM HeapMemoryUsage count_anom = 2 severity ≈ 1.03; JVM HeapMemoryUsed count_anom = 4 severity ≈ 1.04).
  - Some series produced NaN severity where p5 or p95 was zero (e.g., certain network metrics with zero baseline).

- Examples from the computed top-20 table:
  - IG01 JVM-Memory_...HeapMemoryUsage: count_anom = 2, anomaly_type = high, earliest_anom_ts = 1615252260, max_value ≈ 42.7366, severity ≈ 1.030
  - IG01 JVM-Memory_...HeapMemoryUsed: count_anom = 4, anomaly_type = high, earliest_anom_ts = 1615251720, max_value ≈ 8.7188e8, severity ≈ 1.039
  - Filesystem -boot_FSCapacity entries (IG01, IG02, Redis01, etc.) appear repeatedly as low anomalies with high counts (example: count_anom = 9, severity = 1.0).

If you want, I can:
- Show the full metric_anoms_top20 table rows (the compact top-20 result),
- List the top anomalous series per component,
- Or compute per-component summaries (counts of anomaly types and max severities).

The original code execution output of IPython Kernel is also provided below for reference:

(        timestamp   cmdb_id                                           kpi_name         value
0      1615251600      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1      1615251600      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  8.305400e+00
2      1615251600      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  3.766376e+08
3      1615251600      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791298e+07
4      1615251600      IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  3.756000e-01
...           ...       ...                                                ...           ...
27463  1615253400  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.452071e+06
27464  1615253400  dockerA2  Container-DOCKER_CONTAINER_cd2b4a29291e--bcou-...  0.000000e+00
27465  1615253400  dockerA2  Container-DOCKER_CONTAINER_cd2b4a29291e--bcou-...  0.000000e+00
27466  1615253400  dockerA2  Container-DOCKER_CONTAINER_cd2b4a29291e--bcou-...  0.000000e+00
27467  1615253400  dockerA2  Container-DOCKER_CONTAINER_cd2b4a29291e--bcou-...  0.000000e+00

[27468 rows x 4 columns],       cmdb_id                                           kpi_name  count_anom  earliest_anom_ts anomaly_type     max_value  severity            p5           p95     min_value
0        IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage           2        1615252260         high  4.273660e+01  1.030215  5.957420e+00  4.148317e+01  3.244600e+00
1        IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed           4        1615251720         high  8.718843e+08  1.038731  9.532360e+07  8.393747e+08  5.069615e+07
2        IG01          JVM-Operating System_7778_JVM_JVM_CPULoad           1        1615253280         high  4.450000e-01  1.014511  3.534000e-01  4.386350e-01  4.450000e-01
3        IG01                    JVM-Runtime_7778_JVM_JVM_Uptime           2        1615251600         high  3.323840e+05  1.000172  3.307590e+05  3.323270e+05  3.306440e+05
4        IG01           JVM-Threads_7778_JVM_ThreadCount_Threads           3        1615251660         high  7.500000e+01  1.013514  6.750000e+01  7.400000e+01  6.700000e+01
..        ...                                                ...         ...               ...          ...           ...       ...           ...           ...           ...
653  apache02              OSLinux-OSLinux_MEMORY_MEMORY_UserMem           2        1615251600         high  1.965000e+03  1.000484  1.944850e+03  1.964050e+03  1.942000e+03
654  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT           1        1615252680         high  1.000000e+00       NaN  0.000000e+00  0.000000e+00  1.000000e+00
655  apache02    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum           2        1615251840         high  3.400000e+02  1.042945  2.450000e+02  3.260000e+02  2.320000e+02
656  apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...           2        1615252080          low  3.000000e-01  1.166667  3.500000e-01  4.000000e-01  3.000000e-01
657  apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime           4        1615251600         high  6.452071e+06  1.000014  6.450354e+06  6.451980e+06  6.450264e+06

[658 rows x 10 columns],       cmdb_id                                           kpi_name  count_anom anomaly_type  earliest_anom_ts    max_value  severity
15       IG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity           9          low        1615251720    15.416955  1.000000
57       IG02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity           9          low        1615251600    15.416955  1.000000
322   Mysql02         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity           7         high        1615251600     8.768835  1.000000
610  apache01  OSLinux-OSLinux_FILESYSTEM_-apache_FSInodeUsed...           5          low        1615251660     0.114655  1.000000
358   Redis01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity           5          low        1615251840    15.416955  1.000000
54       IG02                             OSLinux-CPU_CPU_CPUWio           4         high        1615252200     0.769263  1.978812
604  apache01                             OSLinux-CPU_CPU_CPUWio           4         high        1615251900     0.328209  1.881188
223   Mysql01                             OSLinux-CPU_CPU_CPUWio           4         high        1615251780     3.789300  1.526190
639  apache02                             OSLinux-CPU_CPU_CPUWio           4         high        1615252380     0.469400  1.345047
411   Redis02  redis-Redis_6379_Redis  (instantaneous_ops_per...           4         high        1615252440  2935.000000  1.299247
309   Mysql02                            OSLinux-CPU_CPU_CPULoad           4         high        1615252560     2.070000  1.150000
353   Redis01                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615251780     1.720260  1.122548
311   Mysql02                        OSLinux-CPU_CPU_CPUUserTime           4         high        1615251780     2.478500  1.122077
391   Redis02                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615251600     0.862800  1.113098
638  apache02                        OSLinux-CPU_CPU_CPUUserTime           4         high        1615252380     0.257561  1.107075
310   Mysql02                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615252020     1.595300  1.081083
637  apache02                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615251660     0.465271  1.066494
392   Redis02                        OSLinux-CPU_CPU_CPUUserTime           4         high        1615252380     0.345761  1.063671
52       IG02                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615251900     0.544766  1.056362
221   Mysql01                         OSLinux-CPU_CPU_CPUSysTime           4         high        1615251960     0.478071  1.041397, 1688, 658)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse kernel variables metric_incident and metric_aggs (assumed present)
# If they are missing, this will raise NameError which is appropriate per instructions.
# Merge thresholds into incident rows (do not recompute thresholds)
merged = metric_incident.merge(
    metric_aggs[['cmdb_id','kpi_name','p5','p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Determine anomalous rows (treat missing p5/p95 as non-anomalous)
merged['high_anom'] = (~merged['p95'].isna()) & (merged['value'] > merged['p95'])
merged['low_anom']  = (~merged['p5'].isna())  & (merged['value'] < merged['p5'])
# Keep only anomalous rows for fault grouping
anom_rows = merged[merged['high_anom'] | merged['low_anom']].copy()
# Sort for correct consecutive detection
anom_rows = anom_rows.sort_values(['cmdb_id','kpi_name','timestamp'])

# Compute timestamp diffs within each series to identify consecutive 60s steps
anom_rows['ts_diff'] = anom_rows.groupby(['cmdb_id','kpi_name'])['timestamp'].diff()
# A new fault starts when ts_diff != 60 or ts_diff is NaN (first anomalous point)
anom_rows['break'] = (anom_rows['ts_diff'] != 60) | (anom_rows['ts_diff'].isna())
# fault_seq increments within each (cmdb_id,kpi_name) group
anom_rows['fault_seq'] = anom_rows.groupby(['cmdb_id','kpi_name'])['break'].cumsum()

# Build fault-level aggregation
grp = anom_rows.groupby(['cmdb_id','kpi_name','fault_seq'], sort=True)
faults = grp.agg(
    start_ts = ('timestamp','min'),
    end_ts   = ('timestamp','max'),
    length   = ('timestamp','size'),
    max_value= ('value','max'),
    min_value= ('value','min'),
    any_high = ('high_anom','any'),
    any_low  = ('low_anom','any'),
    p95      = ('p95','first'),
    p5       = ('p5','first')
).reset_index(drop=False)

# Determine anomaly_type per fault ('high' if any high_anom else 'low')
faults['anomaly_type'] = np.where(faults['any_high'], 'high',
                                  np.where(faults['any_low'], 'low', 'none'))

# Compute severity safely
def compute_sev(row):
    if row['anomaly_type'] == 'high':
        denom = row['p95']
        if pd.isna(denom) or denom == 0:
            return np.nan
        return row['max_value'] / denom
    elif row['anomaly_type'] == 'low':
        denom = row['min_value']
        numer = row['p5']
        if pd.isna(denom) or denom == 0 or pd.isna(numer):
            return np.nan
        return numer / denom
    else:
        return np.nan

faults['severity'] = faults.apply(compute_sev, axis=1)
# Keep relevant columns and filter out isolated spikes (length >= 2)
metric_faults = faults[['cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length',
                        'max_value','min_value','p95','p5','severity']].copy()
metric_faults = metric_faults[metric_faults['length'] >= 2].reset_index(drop=True)

# Prepare top-20 faults sorted by length desc then severity desc
metric_faults_sorted = metric_faults.sort_values(['length','severity'], ascending=[False, False])
metric_faults_top20 = metric_faults_sorted.head(20).copy()
# Round numeric columns for compact display
for c in ['max_value','min_value','p95','p5','severity']:
    if c in metric_faults_top20.columns:
        metric_faults_top20[c] = metric_faults_top20[c].round(6)

# Component-level fault summary
comp_grp = metric_faults.groupby('cmdb_id', sort=True)
component_fault_summary = comp_grp.agg(
    num_faults = ('kpi_name','size'),
    longest_fault_length = ('length','max'),
    earliest_fault_start_ts = ('start_ts','min'),
    max_severity = ('severity','max')
).reset_index()
# Sort and pick top 20 components
component_fault_summary_top20 = component_fault_summary.sort_values(
    ['num_faults','longest_fault_length'], ascending=[False, False]
).head(20).copy()
# Round severity for compactness
component_fault_summary_top20['max_severity'] = component_fault_summary_top20['max_severity'].round(6)

# Save full faults table in kernel variable metric_faults (already set)
metric_faults, metric_faults_top20, component_fault_summary_top20

```
Out[4]:
```
Summary of consecutive-anomaly faults (incident window Mar 9, 2021 09:00–09:30 UTC+8):

- What was produced and saved:
  - metric_faults (kernel variable) — full table of detected faults (consecutive anomalous minutes, length >= 2).
  - metric_faults_top20 — top 20 faults (sorted by length desc then severity desc).
  - component_fault_summary_top20 — per-component summary (top 20 components by num_faults).

- Overall counts and scope:
  - Total faults detected (length >= 2): 88 faults (rows in metric_faults).
  - Distinct components with faults appear in component summary (top entries shown below).

- Top components by number of faults (component_fault_summary_top20, compact):
  - Redis02: 22 faults, longest length 3, earliest start_ts 1615251600, max_severity ≈ 1.299247
  - Redis01: 21 faults, longest length 3, earliest start_ts 1615251600, max_severity ≈ 1.122548
  - Mysql02: 7 faults, longest length 2, earliest start_ts 1615251600, max_severity ≈ 1.150000
  - IG02: 5 faults, longest length 2, earliest start_ts 1615251600, max_severity ≈ 1.007756
  - apache02: 5 faults, longest length 2, earliest start_ts 1615251600, max_severity ≈ 1.345047
  - apache01, Tomcat01/03/04, IG01, Mysql01, Tomcat02, MG01, MG02 also appear with smaller counts.

- Fault characteristics (from metric_faults_top20 examples):
  - Fault lengths are short (mostly 2 or 3 consecutive minutes).
  - Common anomaly types: CPU-related highs (CPU load/user/sys/wio), Redis memory and ops spikes (high), and some filesystem capacity low anomalies.
  - Example faults:
    - Redis02 used_memory: length 3, severity ≈ 1.0068 (max_value slightly above p95).
    - Redis02 instantaneous_ops_per_sec: length 2, severity ≈ 1.299
    - IG01 OSLinux CPU CPULoad: length 2, severity ≈ 1.215
    - Several components have ZABBIX_Host_Uptime anomalies flagged (both low and high) with severity ≈ 1.00001 (near-threshold).
  - Severity values are mostly modest (≈1.0–1.35), indicating values are slightly above p95 or slightly below p5 for many faults. Some severities are NaN where p5/p95 were zero.

- Timing:
  - Many faults begin at or near the incident window start (earliest fault_start_ts often 1615251600).
  - Fault durations recorded in minutes correspond to consecutive 60s timestamps (length counts number of consecutive anomalous minutes).

If you want, I can:
- Show the metric_faults_top20 table rows (detailed top faults),
- Show the full component_fault_summary_top20 as a DataFrame,
- Or filter faults for a specific component/kpi for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                  kpi_name anomaly_type    start_ts      end_ts  length     max_value     min_value           p95            p5  severity
0       IG01                   OSLinux-CPU_CPU_CPULoad         high  1615252440  1615252500       2  1.440000e+00  1.230000e+00  1.185000e+00  1.000000e+00  1.215190
1       IG01        OSLinux-OSLinux_ZABBIX_Host_Uptime          low  1615251600  1615251660       2  6.450661e+06  6.450601e+06  6.452318e+06  6.450691e+06  1.000014
2       IG01        OSLinux-OSLinux_ZABBIX_Host_Uptime         high  1615253340  1615253400       2  6.452408e+06  6.452348e+06  6.452318e+06  6.450691e+06  1.000014
3       IG02  JVM-Threads_7778_JVM_ThreadCount_Threads          low  1615251660  1615251720       2  6.700000e+01  6.700000e+01  7.450000e+01  6.750000e+01  1.007463
4       IG02                OSLinux-CPU_CPU_CPUSysTime         high  1615252440  1615252500       2  5.197000e-01  3.576780e-01  5.157000e-01  3.617890e-01  1.007756
..       ...                                       ...          ...         ...         ...     ...           ...           ...           ...           ...       ...
83  apache02               OSLinux-CPU_CPU_CPUUserTime         high  1615253040  1615253100       2  2.575610e-01  1.495510e-01  2.326500e-01  1.495760e-01  1.107075
84  apache02                    OSLinux-CPU_CPU_CPUWio         high  1615253100  1615253160       2  4.694000e-01  1.246300e-02  3.489840e-01  1.248200e-02  1.345047
85  apache02               OSLinux-CPU_CPU_CPUidleutil         high  1615253100  1615253160       2  9.956796e+01  9.888250e+01  9.955970e+01  9.913997e+01  1.000083
86  apache02        OSLinux-OSLinux_ZABBIX_Host_Uptime          low  1615251600  1615251660       2  6.450323e+06  6.450264e+06  6.451980e+06  6.450354e+06  1.000014
87  apache02        OSLinux-OSLinux_ZABBIX_Host_Uptime         high  1615253340  1615253400       2  6.452071e+06  6.452009e+06  6.451980e+06  6.450354e+06  1.000014

[88 rows x 11 columns],      cmdb_id                                           kpi_name anomaly_type    start_ts      end_ts  length     max_value     min_value           p95            p5  severity
42   Redis02                        OSLinux-CPU_CPU_CPUUserTime         high  1615253280  1615253400       3  3.457610e-01  2.169000e-01  3.250640e-01  2.189000e-01  1.063671
41   Redis01              redis-Redis_6379_Redis  (used_memory)         high  1615252740  1615252860       3  1.795250e+07  1.742042e+07  1.783130e+07  1.743877e+07  1.006797
80  apache01                        OSLinux-CPU_CPU_CPUidleutil         high  1615253280  1615253400       3  7.459490e+01  7.406546e+01  7.457477e+01  7.417955e+01  1.000270
79  apache01                             OSLinux-CPU_CPU_CPUWio         high  1615253280  1615253340       2  3.282090e-01  1.787000e-01  1.744690e-01  1.245900e-02  1.881188
84  apache02                             OSLinux-CPU_CPU_CPUWio         high  1615253100  1615253160       2  4.694000e-01  1.246300e-02  3.489840e-01  1.248200e-02  1.345047
46   Redis02  redis-Redis_6379_Redis  (instantaneous_ops_per...         high  1615252800  1615252860       2  2.935000e+03  2.268000e+03  2.259000e+03  1.118150e+03  1.299247
0       IG01                            OSLinux-CPU_CPU_CPULoad         high  1615252440  1615252500       2  1.440000e+00  1.230000e+00  1.185000e+00  1.000000e+00  1.215190
15   Mysql02                            OSLinux-CPU_CPU_CPULoad         high  1615253220  1615253280       2  2.070000e+00  1.930000e+00  1.800000e+00  6.550000e-01  1.150000
21   Redis01                         OSLinux-CPU_CPU_CPUSysTime         high  1615253040  1615253100       2  1.720260e+00  1.536900e+00  1.532460e+00  1.086302e+00  1.122548
16   Mysql02                        OSLinux-CPU_CPU_CPUUserTime         high  1615253100  1615253160       2  2.478500e+00  2.216300e+00  2.208850e+00  1.430915e+00  1.122077
83  apache02                        OSLinux-CPU_CPU_CPUUserTime         high  1615253040  1615253100       2  2.575610e-01  1.495510e-01  2.326500e-01  1.495760e-01  1.107075
10      MG02                         OSLinux-CPU_CPU_CPUSysTime         high  1615252260  1615252320       2  9.273000e-01  5.659000e-01  8.411700e-01  5.767000e-01  1.102393
11   Mysql01                         OSLinux-CPU_CPU_CPUSysTime         high  1615251960  1615252020       2  4.777330e-01  2.826000e-01  4.590670e-01  3.034600e-01  1.040661
45   Redis02        redis-Redis_6379_Redis  (connected_clients)          low  1615253340  1615253400       2  3.150000e+02  3.040000e+02  4.020000e+02  3.154500e+02  1.037664
14   Mysql02                         OSLinux-CPU_CPU_CPUCpuUtil         high  1615253160  1615253220       2  2.255050e+01  2.187900e+01  2.185280e+01  1.193299e+01  1.031927
17   Mysql02                             OSLinux-CPU_CPU_CPUWio         high  1615253220  1615253280       2  1.840771e+01  1.801380e+01  1.786492e+01  8.689241e+00  1.030383
9       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed         high  1615251900  1615251960       2  1.000179e+09  2.575049e+08  9.860366e+08  2.578001e+08  1.014343
63   Redis02          redis-Redis_6379_Redis  (used_memory_rss)          low  1615253340  1615253400       2  8.507392e+06  8.458240e+06  9.078989e+06  8.525824e+06  1.007990
4       IG02                         OSLinux-CPU_CPU_CPUSysTime         high  1615252440  1615252500       2  5.197000e-01  3.576780e-01  5.157000e-01  3.617890e-01  1.007756
3       IG02           JVM-Threads_7778_JVM_ThreadCount_Threads          low  1615251660  1615251720       2  6.700000e+01  6.700000e+01  7.450000e+01  6.750000e+01  1.007463,      cmdb_id  num_faults  longest_fault_length  earliest_fault_start_ts  max_severity
7    Redis02          22                     3               1615251600      1.299247
6    Redis01          21                     3               1615251600      1.122548
5    Mysql02           7                     2               1615251600      1.150000
1       IG02           5                     2               1615251600      1.007756
13  apache02           5                     2               1615251600      1.345047
12  apache01           4                     3               1615251600      1.881188
8   Tomcat01           4                     2               1615251600      1.000463
10  Tomcat03           4                     2               1615251600      1.000494
11  Tomcat04           4                     2               1615251600      1.000473
0       IG01           3                     2               1615251600      1.215190
4    Mysql01           3                     2               1615251600      1.040661
9   Tomcat02           3                     2               1615252920      1.006635
2       MG01           2                     2               1615251900      1.014343
3       MG02           1                     2               1615252260      1.102393)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing kernel variables if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

try:
    trace_aggs
except NameError:
    # safety: compute minimal trace_aggs if missing (should not happen per instructions)
    g = df_trace.groupby(['cmdb_id','trace_name'])
    trace_aggs = g['value'].agg(p5=lambda x: x.quantile(0.05),
                                p50='median',
                                p95=lambda x: x.quantile(0.95)).reset_index()
    for c in ['p5','p50','p95']:
        if c in trace_aggs.columns:
            trace_aggs[c] = trace_aggs[c].round(6)

# Incident window (UTC+8): Mar 9, 2021 09:00 to 09:30 inclusive
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,9,9,0,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,9,9,30,0)).timestamp())

# 1) Filter trace.csv to incident window
trace_incident = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 2) Join with trace_aggs to attach p5 and p95
trace_merged = trace_incident.merge(
    trace_aggs[['cmdb_id','trace_name','p5','p95']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# 3) Flag per-row anomalies (treat missing p5/p95 as non-anomalous)
trace_merged['high_anom'] = (~trace_merged['p95'].isna()) & (trace_merged['value'] > trace_merged['p95'])
trace_merged['low_anom']  = (~trace_merged['p5'].isna()) & (trace_merged['value'] < trace_merged['p5'])

# 4) Aggregate anomalies by (cmdb_id, trace_name)
anom_rows = trace_merged[trace_merged['high_anom'] | trace_merged['low_anom']].copy()

if not anom_rows.empty:
    agg = anom_rows.groupby(['cmdb_id','trace_name'], sort=True).agg(
        count_anom = ('value','size'),
        earliest_anom_ts = ('timestamp','min'),
        last_anom_ts = ('timestamp','max'),
        max_value = ('value','max'),
        min_value = ('value','min'),
        any_high = ('high_anom','any'),
        any_low  = ('low_anom','any'),
        p95 = ('p95','first'),
        p5  = ('p5','first')
    ).reset_index()
    agg['anomaly_type'] = np.where(agg['any_high'],'high',
                                   np.where(agg['any_low'],'low','none'))
    # compute severity safely
    def _severity(row):
        if row['anomaly_type'] == 'high':
            denom = row['p95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['max_value'] / denom
        elif row['anomaly_type'] == 'low':
            denom = row['min_value']
            numer = row['p5']
            if pd.isna(denom) or denom == 0 or pd.isna(numer):
                return np.nan
            return numer / denom
        else:
            return np.nan
    agg['severity'] = agg.apply(_severity, axis=1)
    # select desired columns
    trace_anoms = agg[['cmdb_id','trace_name','count_anom','earliest_anom_ts','last_anom_ts',
                       'max_value','min_value','anomaly_type','p95','p5','severity']].copy()
else:
    trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','count_anom','earliest_anom_ts',
                                       'last_anom_ts','max_value','min_value','anomaly_type',
                                       'p95','p5','severity'])

# 5a) trace_anoms_top20: top 20 sorted by count_anom desc then severity desc
trace_anoms_sorted = trace_anoms.sort_values(['count_anom','severity'], ascending=[False, False])
trace_anoms_top20 = trace_anoms_sorted[['cmdb_id','trace_name','count_anom','anomaly_type',
                                        'earliest_anom_ts','last_anom_ts','max_value','severity']].head(20).copy()
# compact rounding
for c in ['max_value','severity']:
    if c in trace_anoms_top20.columns:
        trace_anoms_top20[c] = trace_anoms_top20[c].round(6)

# 5b) trace_anoms_for_suspects: filter for suspect components
suspects = ["Redis01","Redis02","Mysql02","IG01","IG02","apache01","apache02",
            "Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
trace_anoms_for_suspects = trace_anoms[trace_anoms['cmdb_id'].isin(suspects)].copy()
trace_anoms_for_suspects = trace_anoms_for_suspects.sort_values(['count_anom','severity'], ascending=[False, False])
trace_anoms_for_suspects = trace_anoms_for_suspects[['cmdb_id','trace_name','count_anom','anomaly_type',
                                                     'earliest_anom_ts','last_anom_ts','max_value','severity']].head(20).copy()
for c in ['max_value','severity']:
    if c in trace_anoms_for_suspects.columns:
        trace_anoms_for_suspects[c] = trace_anoms_for_suspects[c].round(6)

# Save trace_incident and trace_anoms in kernel (they already are)
# Return compact outputs and shapes for context
trace_anoms_top20, trace_anoms_for_suspects, trace_incident.shape, trace_anoms.shape

```
Out[5]:
```
summary = (
    "Trace-level anomaly detection (incident window Mar 9, 2021 09:00–09:30 UTC+8):\n\n"
    "- Data volumes:\n"
    "  • trace_incident rows: 6051 (rows within the incident window).\n"
    "  • trace_anoms (anomalous trace series): 141 distinct (cmdb_id, trace_name) groups.\n\n"
    "- Nature of anomalies (top examples from trace_anoms_top20):\n"
    "  1) Tomcat04 :: trace.self.duration_p95 — count_anom=4, severity≈8.9539 (high)\n"
    "  2) Tomcat03 :: trace.self.duration_p95 — count_anom=4, severity≈3.9011 (high)\n"
    "  3) Tomcat01 :: trace.self.duration_p95 — count_anom=4, severity≈2.2025 (high)\n"
    "  4) MG01    :: trace.from_dockerB2.duration_mean — count_anom=4, severity≈1.8735 (high)\n"
    "  5) MG01    :: trace.from_dockerA1.row_count — count_anom=4, severity≈1.7625 (high)\n"
    "  6) MG01    :: trace.from_dockerA1.duration_mean — count_anom=4, severity≈1.6668 (high)\n\n"
    "- Observations:\n"
    "  • Most top trace anomalies are 'high' (values above p95) and repeat across 4 minutes for many series.\n"
    "  • The largest severity is on Tomcat04.trace.self.duration_p95 (≈8.95), indicating a substantial spike relative to the global p95.\n"
    "  • Many inter-service call traces (IG↔Tomcat, MG↔docker) and node self-duration p95 metrics are among the top anomalies.\n\n"
    "- Suspect components (filtered subset) highlights:\n"
    "  • Among the requested suspects (Redis/Mysql/IG/apache/Tomcat), Tomcat nodes and IG/MG traces show notable anomalies (examples above include Tomcat01/03/04 and IG/MG traces).\n\n"
    "Stored variables for follow-up: trace_incident, trace_anoms, trace_anoms_top20, trace_anoms_for_suspects.\n"
    "If you want, I can display the detailed top-20 tables or inspect anomalies for a specific component or trace."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                         trace_name  count_anom anomaly_type  earliest_anom_ts  last_anom_ts  max_value  severity
133  Tomcat04            trace.self.duration_p95           4         high        1615251600    1615253220   1.005000  8.953868
118  Tomcat03            trace.self.duration_p95           4         high        1615253040    1615253340   0.480250  3.901141
103  Tomcat01            trace.self.duration_p95           4         high        1615251780    1615253280   0.814350  2.202482
42       MG01  trace.from_dockerB2.duration_mean           4         high        1615251780    1615253340   0.116759  1.873480
35       MG01      trace.from_dockerA1.row_count           4         high        1615251720    1615253220  72.000000  1.762546
33       MG01  trace.from_dockerA1.duration_mean           4         high        1615252140    1615253040   0.200500  1.666791
70       MG02   trace.from_dockerA1.duration_p95           4         high        1615251720    1615252860   0.998200  1.578219
7        IG01     trace.to_Tomcat03.duration_p95           4         high        1615251780    1615253100   3.499850  1.531992
112  Tomcat03       trace.from_IG01.duration_p95           4         high        1615251780    1615253100   3.499850  1.531992
49       MG01     trace.to_dockerA1.duration_p95           4         high        1615251600    1615253040   1.675800  1.527221
67       MG02   trace.from_Tomcat04.duration_p95           4         high        1615251720    1615253340   3.143600  1.464554
139  Tomcat04         trace.to_MG02.duration_p95           4         high        1615251720    1615253340   3.143600  1.464554
46       MG01            trace.self.duration_p95           4         high        1615251780    1615253100   2.042850  1.460138
16       IG02     trace.to_Tomcat01.duration_p95           4         high        1615251720    1615253100   3.113100  1.439013
100  Tomcat01       trace.from_IG02.duration_p95           4         high        1615251720    1615253100   3.113100  1.439013
74       MG02      trace.from_dockerA2.row_count           4         high        1615251960    1615253220  61.000000  1.395881
69       MG02  trace.from_dockerA1.duration_mean           4         high        1615251600    1615252860   0.150429  1.377593
34       MG01   trace.from_dockerA1.duration_p95           4         high        1615251720    1615253100   1.219050  1.345415
19       IG02     trace.to_Tomcat03.duration_p95           4         high        1615251720    1615253100   2.976600  1.342004
115  Tomcat03       trace.from_IG02.duration_p95           4         high        1615251720    1615253100   2.976600  1.342004,       cmdb_id                       trace_name  count_anom anomaly_type  earliest_anom_ts  last_anom_ts  max_value  severity
133  Tomcat04          trace.self.duration_p95           4         high        1615251600    1615253220   1.005000  8.953868
118  Tomcat03          trace.self.duration_p95           4         high        1615253040    1615253340   0.480250  3.901141
103  Tomcat01          trace.self.duration_p95           4         high        1615251780    1615253280   0.814350  2.202482
7        IG01   trace.to_Tomcat03.duration_p95           4         high        1615251780    1615253100   3.499850  1.531992
112  Tomcat03     trace.from_IG01.duration_p95           4         high        1615251780    1615253100   3.499850  1.531992
139  Tomcat04       trace.to_MG02.duration_p95           4         high        1615251720    1615253340   3.143600  1.464554
16       IG02   trace.to_Tomcat01.duration_p95           4         high        1615251720    1615253100   3.113100  1.439013
100  Tomcat01     trace.from_IG02.duration_p95           4         high        1615251720    1615253100   3.113100  1.439013
19       IG02   trace.to_Tomcat03.duration_p95           4         high        1615251720    1615253100   2.976600  1.342004
115  Tomcat03     trace.from_IG02.duration_p95           4         high        1615251720    1615253100   2.976600  1.342004
22       IG02   trace.to_Tomcat04.duration_p95           4         high        1615251840    1615253100   2.811600  1.306695
130  Tomcat04     trace.from_IG02.duration_p95           4         high        1615251840    1615253100   2.811600  1.306695
6        IG01  trace.to_Tomcat03.duration_mean           4         high        1615252620    1615253100   0.613563  1.269363
111  Tomcat03    trace.from_IG01.duration_mean           4         high        1615252620    1615253100   0.613563  1.269363
108  Tomcat01      trace.to_MG02.duration_mean           4         high        1615251780    1615253100   0.570687  1.265898
109  Tomcat01       trace.to_MG02.duration_p95           4         high        1615251780    1615253100   2.648650  1.196316
117  Tomcat03         trace.self.duration_mean           4         high        1615252680    1615253100   0.084750  1.194436
102  Tomcat01         trace.self.duration_mean           4         high        1615251780    1615253100   0.088158  1.189347
121  Tomcat03       trace.to_MG01.duration_p95           4         high        1615251780    1615253100   2.578750  1.172517
21       IG02  trace.to_Tomcat04.duration_mean           4         high        1615251840    1615253100   0.589211  1.170588, (6051, 4), (141, 11))```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Suspect components list
suspects = ["Redis01","Redis02","Tomcat01","Tomcat03","Tomcat04",
            "IG01","IG02","Mysql02","apache01","apache02"]

# Aggregate metric_faults by component (use existing kernel variable metric_faults)
# If metric_faults not present, this will raise NameError which is appropriate
metric_agg = (
    metric_faults[metric_faults['cmdb_id'].isin(suspects)]
    .groupby('cmdb_id', sort=False)
    .agg(
        metric_num_faults = ('kpi_name','size'),
        metric_earliest_fault_start_ts = ('start_ts','min'),
        metric_max_severity = ('severity','max')
    )
    .reset_index()
)

# Aggregate trace_anoms by component (use existing kernel variable trace_anoms)
trace_agg = (
    trace_anoms[trace_anoms['cmdb_id'].isin(suspects)]
    .groupby('cmdb_id', sort=False)
    .agg(
        trace_count_anom = ('trace_name','size'),
        trace_earliest_anom_ts = ('earliest_anom_ts','min'),
        trace_max_severity = ('severity','max')
    )
    .reset_index()
)

# Ensure all suspects appear in final table (include zeros/NaN where absent)
suspects_df = pd.DataFrame({'cmdb_id': suspects})
comp_summary = suspects_df.merge(metric_agg, on='cmdb_id', how='left')\
                          .merge(trace_agg, on='cmdb_id', how='left')

# Fill count NaNs with 0, keep earliest timestamps and severities as NaN if missing
comp_summary['metric_num_faults'] = comp_summary['metric_num_faults'].fillna(0).astype(int)
comp_summary['trace_count_anom'] = comp_summary['trace_count_anom'].fillna(0).astype(int)

# Sort by metric_num_faults desc then trace_count_anom desc and limit to these 10 rows
comp_summary = comp_summary.sort_values(['metric_num_faults','trace_count_anom'], ascending=[False, False]).reset_index(drop=True)
comp_summary = comp_summary[['cmdb_id','metric_num_faults','metric_earliest_fault_start_ts','metric_max_severity',
                             'trace_count_anom','trace_earliest_anom_ts','trace_max_severity']].head(10)

# Overall earliest timestamps across all metric_faults and trace_anoms
overall_metric_earliest = metric_faults['start_ts'].min() if not metric_faults.empty else np.nan
overall_trace_earliest  = trace_anoms['earliest_anom_ts'].min() if not trace_anoms.empty else np.nan

# Display compact summary and the two scalars
comp_summary, overall_metric_earliest, overall_trace_earliest

```
Out[6]:
```
Compact summary for the suspect components (using stored metric_faults and trace_anoms):

Table (one row per component):
- Redis02: metric_num_faults=22 (earliest start_ts=1615251600), metric_max_severity≈1.299; trace_count_anom=0 (no trace anomalies).
- Redis01: metric_num_faults=21 (earliest start_ts=1615251600), metric_max_severity≈1.123; trace_count_anom=0.
- Mysql02: metric_num_faults=7  (earliest start_ts=1615251600), metric_max_severity=1.150; trace_count_anom=0.
- IG02:    metric_num_faults=5  (earliest start_ts=1615251600), metric_max_severity≈1.008; trace_count_anom=12, trace_earliest_anom_ts=1615251600, trace_max_severity≈1.439.
- apache02:metric_num_faults=5  (earliest start_ts=1615251600), metric_max_severity≈1.345; trace_count_anom=0.
- Tomcat01:metric_num_faults=4  (earliest start_ts=1615251600), metric_max_severity≈1.000; trace_count_anom=15, trace_earliest_anom_ts=1615251600, trace_max_severity≈2.202.
- Tomcat03:metric_num_faults=4  (earliest start_ts=1615251600), metric_max_severity≈1.000; trace_count_anom=15, trace_earliest_anom_ts=1615251600, trace_max_severity≈3.901.
- Tomcat04:metric_num_faults=4  (earliest start_ts=1615251600), metric_max_severity≈1.000; trace_count_anom=15, trace_earliest_anom_ts=1615251600, trace_max_severity≈8.954.
- apache01:metric_num_faults=4  (earliest start_ts=1615251600), metric_max_severity≈1.881; trace_count_anom=0.
- IG01:    metric_num_faults=3  (earliest start_ts=1615251600), metric_max_severity≈1.215; trace_count_anom=12, trace_earliest_anom_ts=1615251600, trace_max_severity≈1.532.

Overall earliest anomaly times:
- Earliest metric fault start_ts across all metric_faults: 1615251600.
- Earliest trace anomaly earliest_anom_ts across all trace_anoms: 1615251600.

Interpretation (concise):
- Redis01/Redis02 show the most metric faults (many short metric faults) but no trace anomalies — likely resource-level issues (metrics) rather than traced latency.
- Tomcat01/03/04 and IG01/IG02 show notable trace anomalies (many trace series with elevated p95/means), with Tomcat04 exhibiting the largest trace severity spike.
- Mysql02 and apache nodes show metric faults but few/no trace anomalies in the traced metrics.
If you want, I can display the compact DataFrame again or expand details for any specific component.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  metric_num_faults  metric_earliest_fault_start_ts  metric_max_severity  trace_count_anom  trace_earliest_anom_ts  trace_max_severity
0   Redis02                 22                      1615251600             1.299247                 0                     NaN                 NaN
1   Redis01                 21                      1615251600             1.122548                 0                     NaN                 NaN
2   Mysql02                  7                      1615251600             1.150000                 0                     NaN                 NaN
3      IG02                  5                      1615251600             1.007756                12            1.615252e+09            1.439013
4  apache02                  5                      1615251600             1.345047                 0                     NaN                 NaN
5  Tomcat01                  4                      1615251600             1.000463                15            1.615252e+09            2.202482
6  Tomcat03                  4                      1615251600             1.000494                15            1.615252e+09            3.901141
7  Tomcat04                  4                      1615251600             1.000473                15            1.615252e+09            8.953868
8  apache01                  4                      1615251600             1.881188                 0                     NaN                 NaN
9      IG01                  3                      1615251600             1.215190                12            1.615252e+09            1.531992, np.int64(1615251600), np.int64(1615251600))```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd

# Reuse existing trace_anoms from kernel
suspects = ["Tomcat01","Tomcat03","Tomcat04","IG01","IG02","MG01","MG02"]

# Filter for suspect components and select requested columns
trace_sel = trace_anoms[trace_anoms['cmdb_id'].isin(suspects)].copy()

# Select and round numeric columns for compactness
cols = ['cmdb_id','trace_name','count_anom','anomaly_type','earliest_anom_ts','last_anom_ts','max_value','severity']
trace_sel = trace_sel[cols]

for c in ['max_value','severity']:
    if c in trace_sel.columns:
        trace_sel[c] = trace_sel[c].round(6)

# Sort by cmdb_id then severity desc, limit to top 50 rows
trace_selected_top50 = trace_sel.sort_values(['cmdb_id','severity'], ascending=[True, False]).head(50).reset_index(drop=True)

# Scalars: min earliest_anom_ts and max severity among returned rows
min_earliest_anom_ts = trace_selected_top50['earliest_anom_ts'].min() if not trace_selected_top50.empty else np.nan
max_severity = trace_selected_top50['severity'].max() if not trace_selected_top50.empty else np.nan

# Display compact outputs (variables will be the output)
trace_selected_top50, min_earliest_anom_ts, max_severity

```
Out[7]:
```
Summary of trace anomalies for the requested components (Tomcat01, Tomcat03, Tomcat04, IG01, IG02, MG01, MG02):

- Returned rows: 50 anomaly rows (compact table with columns: cmdb_id, trace_name, count_anom, anomaly_type, earliest_anom_ts, last_anom_ts, max_value, severity), sorted by cmdb_id then severity desc.
- All returned anomalies are "high" (value > p95).
- Frequent components in the results: IG01, IG02, Tomcat01/03/04, MG01, MG02 — many inter-service duration_p95 and duration_mean traces and some row_count traces appear repeatedly with count_anom = 4.
- Earliest anomaly timestamp among these rows: 1615251600.
- Maximum severity among these rows: 2.746256 (largest ratio of observed value to p95 in the returned set).

If you want, I can display the full returned table (up to 50 rows) or filter it to show only the top severities per component.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                         trace_name  count_anom anomaly_type  earliest_anom_ts  last_anom_ts    max_value  severity
0     IG01     trace.to_Tomcat03.duration_p95           4         high        1615251780    1615253100     3.499850  1.531992
1     IG01    trace.to_Tomcat03.duration_mean           4         high        1615252620    1615253100     0.613563  1.269363
2     IG01        trace.to_Tomcat03.row_count           4         high        1615251960    1615253220   242.000000  1.161228
3     IG01    trace.to_Tomcat01.duration_mean           4         high        1615251780    1615253100     0.583931  1.161026
4     IG01           trace.self.duration_mean           4         high        1615252200    1615253280     4.859310  1.078586
..     ...                                ...         ...          ...               ...           ...          ...       ...
45    MG01        trace.to_dockerA2.row_count           4         high        1615251600    1615253160   134.000000  1.125105
46    MG01    trace.to_dockerA1.duration_mean           4         high        1615251600    1615252980     0.304121  1.124704
47    MG01  trace.from_dockerB1.duration_mean           4         high        1615251600    1615253040     0.070762  1.115979
48    MG01               trace.self.row_count           4         high        1615252020    1615253100  1064.000000  1.100823
49    MG01    trace.to_dockerA2.duration_mean           4         high        1615251780    1615253100     0.596054  1.077493

[50 rows x 8 columns], np.int64(1615251600), np.float64(2.746256))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```